# Week 18 -- RLHF & PPO for Language Models -- Workbook

Each exercise is a triple: prompt (markdown) -- hint -- scaffold (raises `NotImplementedError`) -- hidden test (calls the reference solution and asserts).
Run the notebook top-to-bottom; reference cells are wired to `project.solutions` so a clean run goes green.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))
from common.plotting import use_rl_style, RL_PALETTE
from common.seed import seed_everything
use_rl_style(); seed_everything(0)
import torch, torch.nn.functional as F, numpy as np
from project import solutions as S
from project.data import ToyPreferenceDataset, heuristic_reward, encode, decode
from project.model import CharTransformer, RewardModel, TransformerConfig
from project.sft import sft_train
from project.rm import rm_train
from project.ppo import compute_gae, ppo_train, PPOConfig

## Exercise 1 (star) -- Bradley--Terry loss
Implement the pairwise log-likelihood loss: given two reward scores `r_w` and `r_l`, return $-\log\sigma(r_w - r_l)$ averaged over the batch.

> **Hint** (try yourself first): use `F.logsigmoid`, not `torch.log(torch.sigmoid(...))`.

In [ ]:
def bt_loss(r_w, r_l):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
torch.manual_seed(0)
rw = torch.randn(8); rl = torch.randn(8)
ref = S.exercise_1_bt_loss(rw, rl)
assert torch.isfinite(ref).all()
assert ref.item() > 0
print('reference loss =', float(ref))

## Exercise 2 (star) -- token-level KL between two categorical distributions

> **Hint**: KL(P || Q) = sum_v P(v) (log P(v) - log Q(v)).  Use the log-softmax for stability.

In [ ]:
def token_kl(logits_p, logits_q):
    raise NotImplementedError

In [ ]:
torch.manual_seed(1)
lp, lq = torch.randn(4, 7), torch.randn(4, 7)
ref = S.exercise_2_token_kl(lp, lq)
assert ref.shape == (4,)
assert (ref >= -1e-6).all()
print('reference KL per sample =', ref)

## Exercise 3 (star star) -- PPO clipped surrogate

> **Hint**: r_t = exp(log pi_theta - log pi_old).  Take the min of unclipped and clamped ratio times the advantage; mind the leading minus sign.

In [ ]:
def ppo_loss(new_logp, old_logp, adv, clip_eps=0.2):
    raise NotImplementedError

In [ ]:
torch.manual_seed(2)
new_logp = torch.randn(8, 4)
old_logp = new_logp + 0.05 * torch.randn(8, 4)
adv = torch.randn(8, 4)
ref = S.exercise_3_ppo_loss(new_logp, old_logp, adv, 0.2)
assert torch.isfinite(ref)
print('reference PPO loss =', float(ref))

## Exercise 4 (star star) -- GAE
Generalised Advantage Estimation: given rewards (B,T) and bootstrapped values (B,T+1), return advantages and returns.

> **Hint**: iterate t from T-1 down to 0, accumulating delta_t + gamma*lam*last.

In [ ]:
def my_gae(rewards, values, gamma=1.0, lam=0.95):
    raise NotImplementedError

In [ ]:
torch.manual_seed(3)
r = torch.randn(2, 5); v = torch.randn(2, 6)
adv_ref, ret_ref = S.exercise_4_gae(r, v)
assert adv_ref.shape == (2, 5) and ret_ref.shape == (2, 5)
print('adv[0]=', adv_ref[0])

## Exercise 5 (star star) -- adaptive KL controller update

> **Hint**: beta <- beta * (1 + clip(observed/target - 1, -0.2, 0.2) / H).

In [ ]:
def kl_update(beta, observed, target, horizon):
    raise NotImplementedError

In [ ]:
beta_up   = S.exercise_5_kl_update(0.05, 0.20, 0.10, 10)
beta_down = S.exercise_5_kl_update(0.05, 0.05, 0.10, 10)
assert beta_up > 0.05 and beta_down < 0.05
print('beta up   =', beta_up)
print('beta down =', beta_down)

## Exercise 6 (star star) -- length-normalised reward
Penalise verbose responses by subtracting alpha * len(y) from the reward.

> **Hint**: a one-liner; cast lengths to float to keep gradients well-typed.

In [ ]:
def length_norm_reward(reward, length, alpha=0.01):
    raise NotImplementedError

In [ ]:
r = torch.tensor([1.0, 2.0]); L = torch.tensor([10, 80])
out = S.exercise_6_length_normalised(r, L, alpha=0.02)
assert out[1] < out[0]
print('lengths penalised =>', out)

## Exercise 7 (star star star) -- closed-form RLHF optimum (DPO seed)
Return the unnormalised log probability log pi*(y) = log pi_ref(y) + r(y)/beta.

> **Hint**: one line, but understand that inverting it gives next week's DPO loss.

In [ ]:
def optimal_logp(log_pi_ref, reward, beta):
    raise NotImplementedError

In [ ]:
lp = torch.tensor([-1.0, -2.0]); r = torch.tensor([0.5, -0.5])
out = S.exercise_7_optimal_policy_logprob(lp, r, beta=0.1)
expected = torch.tensor([-1.0 + 5.0, -2.0 - 5.0])
assert torch.allclose(out, expected)
print('optimal logp =', out)

## Exercise 8 (star star star) -- *Open-ended:* proxy-vs-gold overoptimisation
Train SFT + RM + PPO on the toy dataset, then sample completions and compare the learned reward to the heuristic gold reward.  A large positive gap signals reward hacking.

> **Hint**: use `S.exercise_8_overoptimisation_score(learned, gold)`.  Push PPO steps higher and watch the gap grow.

In [ ]:
ds = ToyPreferenceDataset(n=32, seed=0)
cfg = TransformerConfig()
policy = CharTransformer(cfg)
rm = RewardModel(CharTransformer(cfg))
sft_train(policy, ds.sft_strings(), steps=60, seed=0)
rm_train(rm, ds.pairs, steps=60, seed=0)
hist, _, _ = ppo_train(policy, rm, ds.prompts(), cfg=PPOConfig(steps=8, batch_size=2, max_new_tokens=12))
prompt_ids = torch.tensor([encode(ds.prompts()[0])])
out = policy.generate(prompt_ids, max_new_tokens=12, greedy=False)
text = decode(out[0])
learned = rm(out, torch.tensor([out.size(1)])).detach()
gold = torch.tensor([heuristic_reward(text)])
gap = S.exercise_8_overoptimisation_score(learned, gold)
print('sample:', text)
print('learned:', float(learned), 'gold:', float(gold), 'gap:', gap)

## Exercise 9 (star star) -- *Meaningful objective:* heuristic-vs-learned correlation
Pearson correlation between the heuristic gold reward and the learned RM scores on a batch of generations.  A drop in correlation after PPO is a clean signal of reward hacking.

> **Hint**: `S.exercise_9_reward_correlation(texts, scores)` does the maths.

In [ ]:
texts, scores = [], []
for p in ds.prompts()[:8]:
    ids = torch.tensor([encode(p)])
    out = policy.generate(ids, max_new_tokens=12, greedy=False)
    text = decode(out[0])
    score = float(rm(out, torch.tensor([out.size(1)])).detach())
    texts.append(text); scores.append(score)
rho = S.exercise_9_reward_correlation(texts, scores)
print(f'Pearson rho (heuristic, learned) = {rho:.3f}')
assert -1.0 <= rho <= 1.0

## Wrap-up
If every cell above ran green, you have a working three-stage RLHF pipeline on a toy character-level model.  The TRL recipe in `project/trl_recipe.py` extends it to Pythia-160M or SmolLM2-360M on a single 24 GB GPU.

Next week we use the closed-form optimum from Exercise 7 to derive Direct Preference Optimization -- the same alignment story without PPO.